# 02 — Ferramentas Customizadas

**Módulo:** EAI_07 — IA Generativa  
**Submódulo:** 05_Agentes  
**Ambiente:** `eai07` (Python 3.11)

---

## O que você vai aprender

- **Ferramentas ricas** — ir além de cálculo simples: busca na web, leitura de arquivos, geração de código
- **Composição de ferramentas** — agente que combina múltiplas ferramentas numa única resposta
- **ToolRunner como framework** — registrar e gerenciar um conjunto maior de ferramentas
- **Tratamento de falhas** — o que acontece quando uma ferramenta retorna erro

---

### Ferramentas deste notebook

| Ferramenta | O que faz |
|---|---|
| `calcular` | Expressões matemáticas seguras (reutilizada do notebook 01) |
| `buscar_web` | Busca no DuckDuckGo e retorna resumo dos resultados |
| `ler_arquivo` | Lê arquivos `.py`, `.md`, `.txt` do projeto |
| `listar_arquivos` | Lista arquivos de um diretório com filtro por extensão |
| `gerar_codigo` | LLM especializado em Python gera código sob demanda |

## Setup

In [1]:
import sys, os, json, math, re
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

from shared.tool_runner import executar_com_tools, ToolRunner

llm = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com'
)
LLM_MODEL    = os.getenv('LLM_MODEL', 'deepseek-chat')
PROJETO_BASE = os.path.abspath('../..')
EAI07_BASE   = os.path.abspath('..')

print(f'LLM     : {LLM_MODEL}')
print(f'Projeto : {PROJETO_BASE}')

LLM     : deepseek-chat
Projeto : C:\Users\Jorge Maques\Documents\Especialista_em_AI


---
## 1. Calculadora (reutilizada)

Mesma implementação do notebook 01 — copiada para este ser independente.

In [2]:
def calcular(expressao: str) -> str:
    """Avalia expressões matemáticas com segurança."""
    permitidos = {
        'sqrt': math.sqrt, 'log': math.log, 'log10': math.log10,
        'sin': math.sin,   'cos': math.cos, 'tan': math.tan,
        'pi': math.pi,     'e': math.e,     'abs': abs,
        'round': round,    'pow': pow,
    }
    try:
        return str(eval(expressao, {'__builtins__': {}}, permitidos))
    except Exception as ex:
        return f'Erro: {ex}'


SCHEMA_CALCULAR = {
    'type': 'function',
    'function': {
        'name': 'calcular',
        'description': 'Avalia expressões matemáticas. Use para cálculos numéricos.',
        'parameters': {
            'type': 'object',
            'properties': {
                'expressao': {'type': 'string', 'description': 'Expressão Python válida. Ex: "sqrt(144)", "pi * 5**2"'}
            },
            'required': ['expressao']
        }
    }
}

# Teste rápido
print(calcular('sqrt(144) + pi'))

15.141592653589793


---
## 2. Busca na Web

Usa a API HTML do DuckDuckGo — sem chave de API, sem autenticação.  
Parseia os resultados com `BeautifulSoup` e retorna título + snippet dos top resultados.

> **Dependência:** `pip install beautifulsoup4 requests`

In [3]:
try:
    import requests
    from bs4 import BeautifulSoup
    print('requests e beautifulsoup4 disponíveis.')
except ImportError:
    print('Instalando dependências...')
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'beautifulsoup4', '-q'])
    import requests
    from bs4 import BeautifulSoup
    print('Instalado com sucesso.')

requests e beautifulsoup4 disponíveis.


In [4]:
def buscar_web(query: str, max_resultados: int = 4) -> str:
    """
    Busca no DuckDuckGo e retorna título + snippet dos primeiros resultados.
    Não requer chave de API.
    """
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/120.0.0.0 Safari/537.36'
    }
    try:
        url  = f'https://html.duckduckgo.com/html/?q={requests.utils.quote(query)}'
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()

        soup       = BeautifulSoup(resp.text, 'html.parser')
        resultados = []

        for item in soup.select('.result')[:max_resultados]:
            titulo  = item.select_one('.result__title')
            snippet = item.select_one('.result__snippet')
            if titulo and snippet:
                resultados.append(
                    f'• {titulo.get_text(strip=True)}\n  {snippet.get_text(strip=True)}'
                )

        if not resultados:
            return f'Nenhum resultado encontrado para: {query}'
        return f'Resultados para "{query}":\n\n' + '\n\n'.join(resultados)

    except requests.exceptions.Timeout:
        return f'Timeout ao buscar: {query}'
    except Exception as e:
        return f'Erro na busca: {e}'


SCHEMA_BUSCAR_WEB = {
    'type': 'function',
    'function': {
        'name': 'buscar_web',
        'description': 'Busca informações atuais na web via DuckDuckGo. '
                       'Use para notícias recentes, documentação, preços, eventos atuais.',
        'parameters': {
            'type': 'object',
            'properties': {
                'query': {
                    'type': 'string',
                    'description': 'Termos de busca em português ou inglês'
                },
                'max_resultados': {
                    'type': 'integer',
                    'description': 'Número de resultados (padrão: 4, máximo: 6)'
                }
            },
            'required': ['query']
        }
    }
}

# Teste
print(buscar_web('DeepSeek API fine-tuning suporte', max_resultados=2))

Resultados para "DeepSeek API fine-tuning suporte":

• DeepSeek Platform
  JoinDeepSeekAPIplatform to access our AI models, developer resources andAPIdocumentation.

• Fine-Tune DeepSeek Models for Custom Use Cases - SitePoint
  How toFine-TuneDeepSeekModels for Custom Use Cases Select a distilled base model (e.g.,DeepSeek-R1-Distill-Qwen-7B) that fits your GPU budget.


---
## 3. Leitor de Arquivos do Projeto

Permite ao agente ler o conteúdo real de arquivos do projeto — código Python,
documentação Markdown, arquivos de configuração.  

Segurança: restringe leitura aos diretórios do projeto, sem acesso a caminhos absolutos externos.

In [5]:
def ler_arquivo(caminho: str, max_linhas: int = 80) -> str:
    """
    Lê o conteúdo de um arquivo do projeto (.py, .md, .txt, .json).
    O caminho é relativo à raiz do EAI_07 ou do projeto.
    """
    extensoes_permitidas = {'.py', '.md', '.txt', '.json', '.yaml', '.yml', '.toml'}

    for base in [EAI07_BASE, PROJETO_BASE]:
        candidato = Path(base) / caminho
        if candidato.exists():
            break
    else:
        candidato = Path(caminho)
        if not candidato.exists():
            return f'Arquivo não encontrado: {caminho}'

    try:
        candidato.resolve().relative_to(Path(PROJETO_BASE).resolve())
    except ValueError:
        return f'Acesso negado: {caminho} está fora do projeto'

    if candidato.suffix not in extensoes_permitidas:
        return f'Extensão não suportada: {candidato.suffix}'

    try:
        linhas = candidato.read_text(encoding='utf-8').splitlines()
        total  = len(linhas)
        texto  = '\n'.join(linhas[:max_linhas])
        aviso  = f'\n\n[... +{total - max_linhas} linhas omitidas]' if total > max_linhas else ''
        return f'# {candidato.name} ({total} linhas)\n\n{texto}{aviso}'
    except Exception as e:
        return f'Erro ao ler {caminho}: {e}'


def listar_arquivos(caminho: str = '', extensao: str = '') -> str:
    """
    Lista arquivos de um diretório do projeto.
    caminho: subdiretório relativo ao EAI_07 (vazio = raiz do EAI_07)
    extensao: filtro opcional, ex: '.py', '.md'
    """
    base    = Path(EAI07_BASE) / caminho if caminho else Path(EAI07_BASE)
    ignorar = {'.git', '__pycache__', '.ipynb_checkpoints', 'venv', '.venv'}

    if not base.exists():
        return f'Diretório não encontrado: {caminho}'

    arquivos = []
    for item in sorted(base.rglob('*')):
        if any(p in item.parts for p in ignorar):
            continue
        if item.is_file():
            if extensao and item.suffix != extensao:
                continue
            arquivos.append(str(item.relative_to(Path(EAI07_BASE))))

    if not arquivos:
        return f'Nenhum arquivo encontrado em: {caminho or "raiz"}'
    return f'{len(arquivos)} arquivo(s) em "{caminho or "raiz"}":\n' + '\n'.join(arquivos[:30])


SCHEMA_LER_ARQUIVO = {
    'type': 'function',
    'function': {
        'name': 'ler_arquivo',
        'description': 'Lê o conteúdo de um arquivo do projeto EAI_07 (.py, .md, .txt). Caminho relativo ao EAI_07.',
        'parameters': {
            'type': 'object',
            'properties': {
                'caminho'   : {'type': 'string',  'description': 'Caminho relativo ao EAI_07. Ex: "shared/llm_factory.py"'},
                'max_linhas': {'type': 'integer', 'description': 'Máximo de linhas (padrão: 80)'}
            },
            'required': ['caminho']
        }
    }
}

SCHEMA_LISTAR_ARQUIVOS = {
    'type': 'function',
    'function': {
        'name': 'listar_arquivos',
        'description': 'Lista arquivos de um diretório do projeto EAI_07. Use caminho="shared" para listar o shared/.',
        'parameters': {
            'type': 'object',
            'properties': {
                'caminho' : {'type': 'string', 'description': 'Subdiretório relativo ao EAI_07. Ex: "shared", "03_RAG"'},
                'extensao': {'type': 'string', 'description': 'Filtro de extensão. Ex: ".py", ".md"'}
            },
            'required': []
        }
    }
}

# Testes
print(listar_arquivos('shared', '.py'))
print()
print(ler_arquivo('shared/llm_factory.py', max_linhas=10))



2 arquivo(s) em "shared":
shared\llm_factory.py
shared\tool_runner.py

# llm_factory.py (231 linhas)

"""
shared/llm_factory.py
Factory provider-agnóstica compartilhada por todos os módulos do EAI_07.

Como usar nos notebooks:
    import sys, os
    sys.path.append(os.path.abspath('..'))          # aponta para EAI_07/
    from shared.llm_factory import chat, get_provider_info


[... +221 linhas omitidas]


---
## 4. Gerador de Código

Uma ferramenta que chama o LLM internamente — especializado em gerar código Python.

Isso mostra um padrão importante: **ferramentas podem usar LLMs**.  
O agente principal decide quando delegar para um sub-LLM especializado.

In [6]:
SYSTEM_CODER = """\
Você é um especialista em Python. Quando solicitado, gere código Python limpo,
funcional e comentado. Retorne APENAS o código, sem explicações antes ou depois.
Use docstrings nas funções. Prefira código idiomático e legível.
"""


def gerar_codigo(descricao: str, contexto: str = '') -> str:
    """
    Usa um LLM especializado para gerar código Python.
    descricao: o que o código deve fazer
    contexto: informações adicionais (bibliotecas disponíveis, restrições)
    """
    prompt = descricao
    if contexto:
        prompt += f'\n\nContexto adicional: {contexto}'

    try:
        resp = llm.chat.completions.create(
            model       = LLM_MODEL,
            messages    = [
                {'role': 'system', 'content': SYSTEM_CODER},
                {'role': 'user',   'content': prompt},
            ],
            temperature = 0.2,
        )
        codigo = resp.choices[0].message.content.strip()
        # Remove blocos markdown se presentes
        codigo = re.sub(r'^```python\s*|^```\s*|\s*```$', '', codigo, flags=re.MULTILINE).strip()
        return codigo
    except Exception as e:
        return f'Erro ao gerar código: {e}'


SCHEMA_GERAR_CODIGO = {
    'type': 'function',
    'function': {
        'name': 'gerar_codigo',
        'description': 'Gera código Python para uma tarefa específica. '
                       'Use quando o usuário pedir implementações, scripts ou funções.',
        'parameters': {
            'type': 'object',
            'properties': {
                'descricao': {
                    'type': 'string',
                    'description': 'Descrição clara do que o código deve fazer'
                },
                'contexto': {
                    'type': 'string',
                    'description': 'Contexto opcional: bibliotecas disponíveis, restrições, exemplos'
                }
            },
            'required': ['descricao']
        }
    }
}

# Teste
print(gerar_codigo('Função que recebe uma lista de números e retorna média, mediana e desvio padrão'))

import math
from typing import List, Tuple


def calcular_estatisticas(numeros: List[float]) -> Tuple[float, float, float]:
    """
    Calcula a média, mediana e desvio padrão de uma lista de números.
    
    Args:
        numeros: Lista de números para análise estatística.
        
    Returns:
        Tupla contendo (média, mediana, desvio padrão).
        
    Raises:
        ValueError: Se a lista estiver vazia.
    """
    if not numeros:
        raise ValueError("A lista de números não pode estar vazia")
    
    # Cálculo da média
    media = sum(numeros) / len(numeros)
    
    # Cálculo da mediana
    numeros_ordenados = sorted(numeros)
    n = len(numeros_ordenados)
    meio = n // 2
    
    if n % 2 == 0:
        mediana = (numeros_ordenados[meio - 1] + numeros_ordenados[meio]) / 2
    else:
        mediana = numeros_ordenados[meio]
    
    # Cálculo do desvio padrão
    soma_quadrados = sum((x - media) ** 2 for x in numeros)
    variancia = soma_quadrados / len(numeros)

---
## 5. Agente com Todas as Ferramentas

Registra todas as ferramentas no `ToolRunner` e testa cenários que combinam múltiplas delas.

In [7]:
SYSTEM_AGENTE = """\
Você é um assistente técnico avançado do projeto EAI_07 — IA Generativa.
Você tem acesso a várias ferramentas e deve usá-las de forma inteligente:

- calcular      → qualquer operação matemática
- buscar_web    → informações atuais, documentação, notícias
- ler_arquivo   → conteúdo de arquivos do projeto
- listar_arquivos → estrutura de diretórios do projeto
- gerar_codigo  → implementações Python sob demanda

Combine ferramentas quando necessário. Seja preciso e técnico.
"""

agente = ToolRunner(system=SYSTEM_AGENTE, verbose=True)
agente.registrar(SCHEMA_CALCULAR,        calcular)
agente.registrar(SCHEMA_BUSCAR_WEB,      buscar_web)
agente.registrar(SCHEMA_LER_ARQUIVO,     ler_arquivo)
agente.registrar(SCHEMA_LISTAR_ARQUIVOS, listar_arquivos)
agente.registrar(SCHEMA_GERAR_CODIGO,    gerar_codigo)

print(agente)

ToolRunner(tools=['calcular', 'buscar_web', 'ler_arquivo', 'listar_arquivos', 'gerar_codigo'])


In [8]:
# Teste 1: leitura de arquivo + geração de código
print('='*60)
print('TESTE 1 — Ler arquivo e gerar código')
print('='*60)
pergunta = (
    'Liste os arquivos Python do diretório shared/. '
    'Depois gere uma função Python que demonstre como usar o llm_factory.py '
    'para fazer uma chamada de chat simples.'
)
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')

TESTE 1 — Ler arquivo e gerar código
👤 Liste os arquivos Python do diretório shared/. Depois gere uma função Python que demonstre como usar o llm_factory.py para fazer uma chamada de chat simples.

[iter 1] 1 tool call(s) — OpenAI
  -> listar_arquivos({'caminho': 'shared', 'extensao': '.py'})
     = 2 arquivo(s) em "shared":
shared\llm_factory.py
shared\tool_runner.py
[iter 2] 1 tool call(s) — OpenAI
  -> ler_arquivo({'caminho': 'shared/llm_factory.py'})
     = # llm_factory.py (231 linhas)

"""
shared/llm_factory.py
Factory provider-agnóstica compartilhada por todos os módulos do EAI_07.

Como usar nos notebooks:
    import sys, os
    sys.path.append(os.path.abspath('..'))          # aponta para EAI_07/
    from shared.llm_factory import chat, get_provider_info

Como usar no projeto Assistente_Tecnico_IA:
    import sys, os
    sys.path.append(os.path.abspath('../../../'))   # aponta para EAI_07/
    from shared.llm_factory import chat

Providers suportados:
    deepseek  → API compa

In [9]:
# Teste 2: busca na web + cálculo
print('='*60)
print('TESTE 2 — Busca na web + cálculo')
print('='*60)
pergunta = (
    'Busque o preço atual do dólar em relação ao real. '
    'Depois calcule quanto ficaria R$ 500 em dólares.'
)
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')

TESTE 2 — Busca na web + cálculo
👤 Busque o preço atual do dólar em relação ao real. Depois calcule quanto ficaria R$ 500 em dólares.

[iter 1] 1 tool call(s) — OpenAI
  -> buscar_web({'query': 'cotação dólar real hoje preço atual', 'max_resultados': 4})
     = Nenhum resultado encontrado para: cotação dólar real hoje preço atual
[iter 2] 1 tool call(s) — OpenAI
  -> buscar_web({'query': 'dólar hoje cotação real brasileiro', 'max_resultados': 4})
     = Nenhum resultado encontrado para: dólar hoje cotação real brasileiro
[iter 3] 1 tool call(s) — OpenAI
  -> buscar_web({'query': 'USD BRL exchange rate today', 'max_resultados': 4})
     = Resultados para "USD BRL exchange rate today":

• 1 USD to BRL - US Dollars to Brazilian Reais Exchange Rate - Xe
  Get the latest 1 US Dollar to Brazilian Realratefor FREE with the original Universal Currency Converter. Setratealerts forUSDtoBRLand learn more about US Dollars and Brazilian Reais from XE - the Currency Authority.

• USD/BRL Currency Ex

In [10]:
# Teste 3: ler arquivo e explicar
print('='*60)
print('TESTE 3 — Ler e explicar arquivo')
print('='*60)
pergunta = 'Leia o arquivo shared/tool_runner.py e explique em 3 pontos o que a função executar_com_tools faz.'
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')


TESTE 3 — Ler e explicar arquivo
👤 Leia o arquivo shared/tool_runner.py e explique em 3 pontos o que a função executar_com_tools faz.

[iter 1] 1 tool call(s) — OpenAI
  -> ler_arquivo({'caminho': 'shared/tool_runner.py'})
     = # tool_runner.py (301 linhas)

"""
shared/tool_runner.py
Executor de function calling compatível com todos os providers,
com tratamento robusto do formato DSML do DeepSeek.

O DeepSeek às vezes retorna tool calls assim em vez do padrão OpenAI:
    <|DSML|function_calls>
    <|DSML|invoke name="nome_funcao">
    <|DSML|parameter name="param">valor</|DSML|parameter>
    </|DSML|invoke>
    </|DSML|function_calls>

Problemas tratados:
    1. DSML pode aparecer em qualquer rodada (não só na primeira)
    2. DSML pode vir misturado com texto ("Vou listar... <DSML>...")
    3. O modelo pode usar nomes de parâmetros diferentes dos definidos
       ex: "diretorio" em vez de "caminho"

Uso:
    from shared.tool_runner import executar_com_tools, ToolRunner
"""

import j

In [11]:
# Teste 4: geração de código
print('='*60)
print('TESTE 4 — Gerar código')
print('='*60)
pergunta = 'Gere um exemplo mínimo e funcional de uso da função executar_com_tools com uma ferramenta de soma.'
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')


TESTE 4 — Gerar código
👤 Gere um exemplo mínimo e funcional de uso da função executar_com_tools com uma ferramenta de soma.

[iter 1] 1 tool call(s) — OpenAI
  -> listar_arquivos({'caminho': 'shared'})
     = 3 arquivo(s) em "shared":
shared\llm_factory.py
shared\requirements.txt
shared\tool_runner.py
[iter 2] 1 tool call(s) — OpenAI
  -> ler_arquivo({'caminho': 'shared/tool_runner.py'})
     = # tool_runner.py (301 linhas)

"""
shared/tool_runner.py
Executor de function calling compatível com todos os providers,
com tratamento robusto do formato DSML do DeepSeek.

O DeepSeek às vezes retorna tool calls assim em vez do padrão OpenAI:
    <|DSML|function_calls>
    <|DSML|invoke name="nome_funcao">
    <|DSML|parameter name="param">valor</|DSML|parameter>
    </|DSML|invoke>
    </|DSML|function_calls>

Problemas tratados:
    1. DSML pode aparecer em qualquer rodada (não só na primeira)
    2. DSML pode vir misturado com texto ("Vou listar... <DSML>...")
    3. O modelo pode usar nomes

---
## 6. Tratamento de Falhas

O agente deve lidar graciosamente quando uma ferramenta falha.
Testamos dois cenários: arquivo inexistente e busca sem resultados.

In [12]:
# Teste de robustez: ferramenta retorna erro
print('='*60)
print('TESTE 4 — Recuperação de erro')
print('='*60)
pergunta = 'Leia o arquivo shared/arquivo_que_nao_existe.py e me diga o que ele faz.'
print(f'👤 {pergunta}\n')
resp = agente.perguntar(pergunta)
print(f'\n🤖 {resp}')

TESTE 4 — Recuperação de erro
👤 Leia o arquivo shared/arquivo_que_nao_existe.py e me diga o que ele faz.

[iter 1] 1 tool call(s) — OpenAI
  -> listar_arquivos({'caminho': 'shared'})
     = 3 arquivo(s) em "shared":
shared\llm_factory.py
shared\requirements.txt
shared\tool_runner.py
[iter 2] 1 tool call(s) — OpenAI
  -> ler_arquivo({'caminho': 'shared/arquivo_que_nao_existe.py'})
     = Arquivo não encontrado: shared/arquivo_que_nao_existe.py
[iter 3] Resposta final

🤖 Como suspeitava, o arquivo `shared/arquivo_que_nao_existe.py` **não existe** no projeto EAI_07. 

Os arquivos disponíveis na pasta `shared` são:
1. **`llm_factory.py`** - Provavelmente contém uma fábrica para criar instâncias de modelos de linguagem
2. **`requirements.txt`** - Lista de dependências Python do projeto
3. **`tool_runner.py`** - Provavelmente gerencia a execução de ferramentas/utilitários

Pelo nome "arquivo_que_nao_existe.py" (que significa "arquivo que não existe" em português), parece que você estava test

---
## Resumo

| Ferramenta | Padrão | Ponto de atenção |
|---|---|---|
| `calcular` | eval com whitelist | Nunca usar `eval` sem restringir `__builtins__` |
| `buscar_web` | requests + BeautifulSoup | DuckDuckGo HTML pode mudar — tratar exceções |
| `ler_arquivo` | Path + relative_to | Sempre validar que o caminho está dentro do projeto |
| `listar_arquivos` | rglob + filtros | Ignorar pastas de cache e venv |
| `gerar_codigo` | LLM especializado | Ferramenta que chama outro LLM — padrão válido |

### Boas práticas

- **Sempre retornar string** — `tool_runner` espera string como resultado de ferramenta
- **Erros como string** — não lançar exceções dentro da ferramenta; retornar mensagem de erro
- **Descrições precisas** — o LLM decide qual ferramenta usar baseado na `description` do schema
- **`required` correto** — parâmetros sem default devem estar em `required`